In [ ]:
import sys
from pathlib import Path

import pandas as pd


# =========================================================
# 1. Make the project root importable
# =========================================================

PROJECT_ROOT_PATH = Path.cwd().parent

if str(PROJECT_ROOT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_PATH))


# =========================================================
# 2. Import project paths and transformation function
# =========================================================

from configs.config import (
    BRONZE_PATH,
    QUARANTINE_PATH,
    SILVER_PATH,
)

from src.silver_transformation import (
    transform_bronze_to_silver,
)


# =========================================================
# 3. Define input and output paths
# =========================================================

bronze_input_path = (
    BRONZE_PATH
    / "credit_card_transactions_sample.parquet"
)

silver_output_path = (
    SILVER_PATH
    / "credit_card_transactions_clean.parquet"
)

quarantine_output_path = (
    QUARANTINE_PATH
    / "credit_card_transactions_invalid.parquet"
)


# =========================================================
# 4. Display paths
# =========================================================

print("Bronze input:", bronze_input_path)
print("Bronze input exists:", bronze_input_path.exists())
print("Silver output:", silver_output_path)
print("Quarantine output:", quarantine_output_path)


# =========================================================
# 5. Run Bronze-to-Silver transformation
# =========================================================

result = transform_bronze_to_silver(
    bronze_input_path=bronze_input_path,
    silver_output_path=silver_output_path,
    quarantine_output_path=quarantine_output_path,
)


# =========================================================
# 6. Display transformation result
# =========================================================

print("\nSilver transformation result:")

for key, value in result.items():
    print(f"{key}: {value}")


# =========================================================
# 7. Read and verify Silver output
# =========================================================

silver_df = pd.read_parquet(
    silver_output_path
)

quarantine_df = pd.read_parquet(
    quarantine_output_path
)

print("\nSilver verification:")
print("Silver file exists:", silver_output_path.exists())
print("Silver rows:", len(silver_df))
print("Silver columns:", len(silver_df.columns))

print("\nQuarantine verification:")
print(
    "Quarantine file exists:",
    quarantine_output_path.exists(),
)
print("Quarantine rows:", len(quarantine_df))
print(
    "Quarantine columns:",
    len(quarantine_df.columns),
)


# =========================================================
# 8. Reconciliation validation
# =========================================================

bronze_df = pd.read_parquet(
    bronze_input_path
)

assert len(bronze_df) == (
    len(silver_df) + len(quarantine_df)
)

assert result["reconciliation_passed"] is True

print("\nRow reconciliation passed:")
print(
    f"{len(bronze_df)} Bronze rows = "
    f"{len(silver_df)} Silver rows + "
    f"{len(quarantine_df)} Quarantine rows"
)


# =========================================================
# 9. Validate final Silver schema
# =========================================================

expected_silver_columns = [
    "user_id",
    "card_id",
    "transaction_timestamp",
    "transaction_amount",
    "transaction_method",
    "merchant_id",
    "merchant_city",
    "merchant_state",
    "merchant_zip_code",
    "merchant_category_code",
    "transaction_error",
    "is_fraud",
    "_source_file_name",
    "_ingestion_timestamp_utc",
    "_pipeline_run_id",
    "_silver_processed_at_utc",
]

assert silver_df.columns.tolist() == (
    expected_silver_columns
)

assert silver_df["user_id"].notna().all()
assert silver_df["card_id"].notna().all()
assert silver_df[
    "transaction_timestamp"
].notna().all()
assert silver_df[
    "transaction_amount"
].notna().all()
assert silver_df["is_fraud"].notna().all()

print("Silver schema and critical-field validation passed.")


# =========================================================
# 10. Display sample Silver records
# =========================================================

print("\nSilver sample:")
display(silver_df.head(10))


# =========================================================
# 11. Display quarantined records, if any
# =========================================================

if quarantine_df.empty:
    print("\nNo records were quarantined.")
else:
    print("\nQuarantined records:")
    display(
        quarantine_df[
            [
                "user_id",
                "card_id",
                "transaction_amount_raw",
                "transaction_timestamp",
                "is_fraud_raw",
                "quarantine_reason",
            ]
        ].head(20)
    )